## 从neo4j读entity、及其与time / event / locgrid 等node 与 relationship
## 并实现：
### 1.计算time0时间切片中，entity两两之间的距离，距离数据放到neo4j中的entity两两之间新建的关系中的distance属性里
### 2.计算时间切片time0中，event发生地与所有entity的距离

- ### 计划：
  - #### 1.先用pyspark+neo4j connector 实现功能，不优化、质量和运行效率
  - #### 2.优化：删除没必要的relationship，让计算更少、更快
  
  

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from utils.distance_utils import *
import os
os.environ['JAVA_HOME'] = "/home/hadoop/app/java-se-8u41-ri/"
os.environ['PYSPARK_SUBMIT_ARGS'] = "--jars file:////home/hadoop/app/spark-3.1.2-bin-without-hadoop/jars/neo4j-connector-apache-spark_2.12-4.1.0_for_spark_3.jar pyspark-shell"

# standalone - spark://hadoop171:7077 yarn
# spark = (SparkSession.builder.master("yarn").appName("car_event_neo4j")
#             #   .config("spark.yarn.am.memory", "8g")
#             #   .config("spark.yarn.am.cores", "4")
#             #   .config("spark.executor.memory", "8g")\
#             #   .config("spark.executor.cores", "24")\
#             #   .config("spark.exector.instances", "6")\
#             #   .config("spark.num-executors", "100")\
#               .getOrCreate())
spark = (SparkSession.builder.master("local[*]").appName("car_event_neo4j")\
            # .config("spark.driver.port", 7077)
            # .config("spark.driver.host", "192.168.203.171")
            .config("spark.executor.memory", "8g")\
            .config("spark.executor.cores", "12")\
            .config("spark.exector.instances", "6")\
            # .config("spark.num-executors", "100")\
            .config("spark.driver.extraJavaOptions", "-XX:PermSize=1024M")\
            .config("spark.executor.extraJavaOptions", "-verbose:gc -XX:+UseCompressedOops -XX:+PrintGCDetails -XX:+PrintGCDateStamps -XX:CMSInitiatingOccupancyFraction=60 -XX:PermSize=1024M")\
            .config("spark.speculation", "false")\
            .config("spark.sql.broadcastTimeout", "600")\
            .config("spark.sql.shuffle.partitions", "800")\
            .config("spark.default.parallelism", "600")\
            .config("spark.memory.fraction", "0.9")\
            .config("spark.dynamicAllocation.enabled", "true")\
            .config("spark.dynamicAllocation.minExecutors", "1")\
            .config("spark.dynamicAllocation.maxExecutors", "100")\
            .config("spark.dynamicAllocation.executorIdleTimeout", "3s")\
            .config("spark.shuffle.service.enabled", "true")\
            .getOrCreate())

OpenJDK 64-Bit Server VM warning: ignoring option PermSize=1024M; support was removed in 8.0
2022-01-07 11:05:19,423 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
# dataframe show不换行
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [3]:
NEO4J_URL = "bolt://192.168.141.136:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "sdc"

In [4]:
dfr = (spark.read.format("org.neo4j.spark.DataSource")
  .option("url", NEO4J_URL)
  .option("authentication.basic.username", NEO4J_USERNAME)
  .option("authentication.basic.password", NEO4J_PASSWORD)
  .option("partitions", 12)
)
  # .option("labels", "Device")\
  # .load()

### 1.计算time0时间切片中，entity两两之间的距离，距离数据放到neo4j中的entity两两之间新建的关系spatial_location中的distance属性里

In [5]:
%%time

# 围绕时间切片time0 查出 t2g_locgrid 和 ent_grid，并在两者关联的entity之间建立 spatial_location 的关系
ent_ent_grids_df = dfr.option("query", """
    MATCH (e1: entity) - [: locatedin] -> (t2g_locgrid: locgrid) <- [t2g: time2loc] - (t: time {ID: "time0"})
          - [t2e: time2entity] -> (e2: entity) - [: locatedin] -> (ent_grid: locgrid)
          WHERE e1.ID <> e2.ID
    RETURN e1.ID AS ent_id1, e1.name AS ent_name1, e2.ID AS ent_id2, e2.name AS ent_name2, 
           ent_grid.ID AS ent_grid_id, ent_grid.name AS ent_grid_name, 
           t2g_locgrid.ID AS t2g_locgrid_id, t2g_locgrid.name AS t2g_locgrid_name
""").load()

# ent_ent_grids_df.repartition(100)
ent_ent_grids_df.cache()

CPU times: user 6.55 ms, sys: 1.34 ms, total: 7.89 ms
Wall time: 19.8 s


ent_id1,ent_name1,ent_id2,ent_name2,ent_grid_id,ent_grid_name,t2g_locgrid_id,t2g_locgrid_name
entity295,538003936,entity296,538003998,locgrid296,00000111010100101...,locgrid295,00000111010110001...
entity294,538003716,entity296,538003998,locgrid296,00000111010100101...,locgrid294,00000101011110100...
entity293,538003673,entity296,538003998,locgrid296,00000111010100101...,locgrid293,00000101000001011...
entity292,538003639,entity296,538003998,locgrid296,00000111010100101...,locgrid292,00000101111110011...
entity291,538003562,entity296,538003998,locgrid296,00000111010100101...,locgrid291,00000111010110001...
entity290,538003558,entity296,538003998,locgrid296,00000111010100101...,locgrid290,00000101100111011...
entity289,538003501,entity296,538003998,locgrid296,00000111010100101...,locgrid289,00010000101010110...
entity288,538003499,entity296,538003998,locgrid296,00000111010100101...,locgrid288,00010010001001100...
entity287,538003274,entity296,538003998,locgrid296,00000111010100101...,locgrid287,00010010000000101...
entity286,538003242,entity296,538003998,locgrid296,00000111010100101...,locgrid286,00000101000001011...


#### 1.1 计算t2g_locgrid 和 ent_grid之间经纬度偏移量和两个网格间的距离

In [6]:
def cal_grid_shift_and_distance(df, grid_name1, grid_name2):
    """
    计算偏移量和距离，并放进df
    """
    df2 = df.withColumn("lat_shift", cal_lat_shift(F.col(grid_name1), F.col(grid_name2)))\
           .withColumn("lon_shift", cal_lon_shift(F.col(grid_name1), F.col(grid_name2)))\
           .withColumn("distance", F.pow(F.pow(F.col("lat_shift"), 2) + F.pow(F.col("lon_shift"), 2), 0.5))
    return df2

In [7]:
%%time

grids_df = cal_grid_shift_and_distance(ent_ent_grids_df, "ent_grid_name", "t2g_locgrid_name")
grids_df.show(2)

+---------+---------+---------+---------+-----------+--------------------+--------------+--------------------+---------+---------+------------------+
|  ent_id1|ent_name1|  ent_id2|ent_name2|ent_grid_id|       ent_grid_name|t2g_locgrid_id|    t2g_locgrid_name|lat_shift|lon_shift|          distance|
+---------+---------+---------+---------+-----------+--------------------+--------------+--------------------+---------+---------+------------------+
|entity295|538003936|entity296|538003998| locgrid296|00000111010100101...|    locgrid295|00000111010110001...|      269|       83|380.42344827836257|
|entity294|538003716|entity296|538003998| locgrid296|00000111010100101...|    locgrid294|00000101011110100...|    -2718|        5|3843.8324625300725|
+---------+---------+---------+---------+-----------+--------------------+--------------+--------------------+---------+---------+------------------+
only showing top 2 rows

CPU times: user 40.2 ms, sys: 11 ms, total: 51.2 ms
Wall time: 2.87 s


#### 1.2 根据lat与lon的偏移量来判定direction，而不用每个方向都存一下distance。注意：每个判断都要用括号包住

In [8]:
grids_df = grids_df.withColumn("direction", (F.when((F.col("lat_shift") > 0) & (F.col("lon_shift") > 0), "NE")
                                              .when((F.col("lat_shift") > 0) & (F.col("lon_shift") < 0), "NW")
                                              .when((F.col("lat_shift") < 0) & (F.col("lon_shift") > 0), "SE")
                                              .when((F.col("lat_shift") < 0) & (F.col("lon_shift") < 0), "SW")
                                              .otherwise(""))
                                              )
grids_df.show(2)

+---------+---------+---------+---------+-----------+--------------------+--------------+--------------------+---------+---------+------------------+---------+
|  ent_id1|ent_name1|  ent_id2|ent_name2|ent_grid_id|       ent_grid_name|t2g_locgrid_id|    t2g_locgrid_name|lat_shift|lon_shift|          distance|direction|
+---------+---------+---------+---------+-----------+--------------------+--------------+--------------------+---------+---------+------------------+---------+
|entity295|538003936|entity296|538003998| locgrid296|00000111010100101...|    locgrid295|00000111010110001...|      269|       83|380.42344827836257|       NE|
|entity294|538003716|entity296|538003998| locgrid296|00000111010100101...|    locgrid294|00000101011110100...|    -2718|        5|3843.8324625300725|       SE|
+---------+---------+---------+---------+-----------+--------------------+--------------+--------------------+---------+---------+------------------+---------+
only showing top 2 rows



In [9]:
# # If getNumPartitions is greater than 1 you might want to repartition it to 1
# if grids_df.rdd.getNumPartitions() > 1:
#     print("repartition 1")
#     grids_df.repartition(1)

#### 1.3 创建spatial_location(entity 2 grid -- time 2 grid)关系，并设置distance和direction两个属性

In [10]:
# %%time

# # 查找是否存在某一属性的节点：match (g:locgrid) where exists(g.ent_grid_id) return g limit 10
# # 删除某一label的关系及其关联节点：MATCH (s)-[r:spatial_location]->(e) DELETE s, r, e

# # relationship.source.save.mode = Match: 只有节点已存在才会建立关系，target同理
# # node.keys、node.properties、relationship.properties的映射模式都是 -- dataframe的列名: Node中的属性名
# (grids_df.write
#     .format("org.neo4j.spark.DataSource")
#     .option("url", NEO4J_URL)
#     .option("authentication.type", "basic")
#     .option("authentication.basic.username", NEO4J_USERNAME)
#     .option("authentication.basic.password", NEO4J_PASSWORD)
#     .option("batch.size", 2500)
#     .option("transaction.retries", "10")
#     .option("transaction.retry.timeout", "1000")
#     .option("relationship.save.strategy", "keys")
#     .option("relationship", "spatial_location")
#     .option("relationship.source.save.mode", "Match")
#     .option("relationship.source.labels", ":entity")
#     # .option("relationship.source.node.keys", "ID")
#     .option("relationship.source.node.keys", "ent_id1:ID")
#     # .option("relationship.source.node.properties", "ent_grid_id:ID")
#     .option("relationship.properties", "distance:distance,direction:direction")
#     .option("relationship.target.save.mode", "Match")
#     .option("relationship.target.labels", ":entity")
#     # .option("relationship.target.node.keys", "ID")
#     .option("relationship.target.node.keys", "ent_id2:ID")
#     # .option("relationship.target.node.properties", "t2g_locgrid_id:ID")
#     .mode("Overwrite")
#     .save()
# )

### 2.计算时间切片time0中，event发生地与所有entity在time0时的所在grid的距离(注意：locgrid_event是网格)

In [11]:
%%time
# RETURN eve.ID AS eve_id, eve.name AS eve_name, eve_grid.ID AS eve_grid_id, eve_grid.name AS eve_grid_name, t.ID AS tm_id, ent.name AS ent_name, ent.ID AS ent_id, ent_grid.ID AS ent_grid_id, ent_grid.name AS ent_grid_name

# 分开查，放到一条cypher查很耗资源

eve_cypher = """
    MATCH (eve: event) - [loc: locatedin] -> (eve_grid: locgrid_event) RETURN eve.ID AS eve_id, eve.name AS eve_name, eve_grid.ID AS eve_grid_id, eve_grid.name AS eve_grid_name
"""

# ent_ent_grids_df已包含了，无需重复查询 大神博客真牛逼
# ent_cypher = """
#     MATCH (ent_grid: locgrid) <- [t2g: time2loc] - (t: time {ID: "time0"}) - [t2e: time2entity] -> (ent: entity)
#     RETURN ent.ID AS ent_id, ent_grid.ID AS ent_grid_id, ent_grid.name AS ent_grid_name
# """

eve_grid_df = dfr.option("query", eve_cypher).load()
# eve_grid_df.repartition(10)
eve_grid_df.cache()

# ent_grid_df = dfr.option("query", ent_cypher).load()
# # ent_grid_df.repartition(10)
# ent_grid_df.cache()


CPU times: user 3.58 ms, sys: 1.37 ms, total: 4.95 ms
Wall time: 375 ms


eve_id,eve_name,eve_grid_id,eve_grid_name
event0,Tsunami,locgrid_event0,00000101100111001...
event1,earthquake,locgrid_event1,10000101010001101...
event2,storm,locgrid_event2,00000111010001110...
event3,mission,locgrid_event3,00000101010101110...
event4,rescue,locgrid_event4,00010010000100001...


In [12]:
print(eve_grid_df.count())
# print(ent_grid_df.count())

ent_ent_grids_df.count()

5


87912

In [13]:
%%time

ent_event_grids_df = ent_ent_grids_df.select("ent_grid_id", "ent_grid_name", "ent_id1", "ent_name1").join(F.broadcast(eve_grid_df))
ent_event_grids_df.show(5)

+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+
|ent_grid_id|       ent_grid_name|  ent_id1|ent_name1|eve_id|  eve_name|   eve_grid_id|       eve_grid_name|
+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+
| locgrid296|00000111010100101...|entity295|538003936|event0|   Tsunami|locgrid_event0|00000101100111001...|
| locgrid296|00000111010100101...|entity295|538003936|event1|earthquake|locgrid_event1|10000101010001101...|
| locgrid296|00000111010100101...|entity295|538003936|event2|     storm|locgrid_event2|00000111010001110...|
| locgrid296|00000111010100101...|entity295|538003936|event3|   mission|locgrid_event3|00000101010101110...|
| locgrid296|00000111010100101...|entity295|538003936|event4|    rescue|locgrid_event4|00010010000100001...|
+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+
only showing top 5 

In [14]:
ent_event_grids_df.cache()
ent_event_grids_df.count()

439560

#### 2.1 计算escape/complete阈值、是否可能要逃跑/完成任务(may_escape_or_complete:0 为escape，1为complete)、eve_grid 和 ent_grid之间经纬度偏移量和两个网格间的距离

In [15]:
%%time

# if event_name[i]=='Tsunami' or event_name[i]=='earthquake' or event_name[i]=='storm':
#             event_entity_relation='escape'
#             event_threshold=5000

ent_event_grids_df = ent_event_grids_df.withColumn("may_escape_or_complete", F.when(F.col("eve_name").isin("Tsunami", "earthquake", "storm"), 0).otherwise(1))\
                                       .withColumn("event_threshold", F.when(F.col("may_escape_or_complete") == 0, 5000).otherwise(3000))

CPU times: user 8.97 ms, sys: 828 µs, total: 9.8 ms
Wall time: 83.3 ms


In [16]:

# ent_event_grids_df.filter(F.col("may_escape_or_complete") == 0).show(3)
# ent_event_grids_df.filter(F.col("may_escape_or_complete") == 1).show(3)

ent_event_grids_df.filter(F.col("may_escape_or_complete") == F.lit(0)).show(3)
ent_event_grids_df.filter(F.col("may_escape_or_complete") == F.lit(1)).show(3)

+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+----------------------+---------------+
|ent_grid_id|       ent_grid_name|  ent_id1|ent_name1|eve_id|  eve_name|   eve_grid_id|       eve_grid_name|may_escape_or_complete|event_threshold|
+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+----------------------+---------------+
| locgrid296|00000111010100101...|entity295|538003936|event0|   Tsunami|locgrid_event0|00000101100111001...|                     0|           5000|
| locgrid296|00000111010100101...|entity295|538003936|event1|earthquake|locgrid_event1|10000101010001101...|                     0|           5000|
| locgrid296|00000111010100101...|entity295|538003936|event2|     storm|locgrid_event2|00000111010001110...|                     0|           5000|
+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+----

In [17]:
%%time
# 计算偏移量和距离
ent_event_grids_df = cal_grid_shift_and_distance(ent_event_grids_df, "eve_grid_name", "ent_grid_name")

CPU times: user 6.84 ms, sys: 946 µs, total: 7.79 ms
Wall time: 67.5 ms


#### 2.2 计算是否真的需要逃跑/完成任务

In [18]:
# need_escape_or_complete: True 为需要，False为不需要
ent_event_grids_df = ent_event_grids_df.withColumn("need_escape_or_complete", F.when((F.col("distance") < F.col("event_threshold")), True).otherwise(False))
ent_event_grids_df.filter(F.col("need_escape_or_complete")).show(3)
ent_event_grids_df.filter(~F.col("need_escape_or_complete")).show(3)

+-----------+--------------------+---------+---------+------+--------+--------------+--------------------+----------------------+---------------+---------+---------+------------------+-----------------------+
|ent_grid_id|       ent_grid_name|  ent_id1|ent_name1|eve_id|eve_name|   eve_grid_id|       eve_grid_name|may_escape_or_complete|event_threshold|lat_shift|lon_shift|          distance|need_escape_or_complete|
+-----------+--------------------+---------+---------+------+--------+--------------+--------------------+----------------------+---------------+---------+---------+------------------+-----------------------+
| locgrid296|00000111010100101...|entity295|538003936|event0| Tsunami|locgrid_event0|00000101100111001...|                     0|           5000|     1849|     1594|2614.8808768278527|                   true|
| locgrid296|00000111010100101...|entity295|538003936|event2|   storm|locgrid_event2|00000111010001110...|                     0|           5000|      139|      183

+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+----------------------+---------------+---------+---------+-----------------+-----------------------+
|ent_grid_id|       ent_grid_name|  ent_id1|ent_name1|eve_id|  eve_name|   eve_grid_id|       eve_grid_name|may_escape_or_complete|event_threshold|lat_shift|lon_shift|         distance|need_escape_or_complete|
+-----------+--------------------+---------+---------+------+----------+--------------+--------------------+----------------------+---------------+---------+---------+-----------------+-----------------------+
| locgrid296|00000111010100101...|entity295|538003936|event1|earthquake|locgrid_event1|10000101010001101...|                     0|           5000|   -28591|      571|40433.77996180916|                  false|
| locgrid296|00000111010100101...|entity295|538003936|event3|   mission|locgrid_event3|00000101010101110...|                     1|           3000|     4311|   

In [19]:
# 过滤出需要逃跑或完成任务的数据，然后写进neo4j 单机neo4j是真的慢
need_escape_df = ent_event_grids_df.filter((F.col("may_escape_or_complete") == 0) & F.col("need_escape_or_complete"))
need_complete_df = ent_event_grids_df.filter((F.col("may_escape_or_complete") == 1) & F.col("need_escape_or_complete"))

In [20]:
# %%time

# (need_escape_df.write
#     .format("org.neo4j.spark.DataSource")
#     .option("url", NEO4J_URL)
#     .option("authentication.type", "basic")
#     .option("authentication.basic.username", NEO4J_USERNAME)
#     .option("authentication.basic.password", NEO4J_PASSWORD)
#     .option("batch.size", 2500)
#     .option("transaction.retries", "10")
#     .option("relationship.save.strategy", "keys")
#     .option("relationship", "escape")
#     .option("relationship.source.save.mode", "Match")
#     .option("relationship.source.labels", ":entity")
#     .option("relationship.source.node.keys", "ent_id1:ID")
#     .option("relationship.properties", "distance")
#     .option("relationship.target.save.mode", "Match")
#     .option("relationship.target.labels", ":event")
#     .option("relationship.target.node.keys", "eve_id:ID")
#     .mode("Overwrite")
#     .save()
# )

In [21]:
# %%time

# (need_complete_df.write
#     .format("org.neo4j.spark.DataSource")
#     .option("url", NEO4J_URL)
#     .option("authentication.type", "basic")
#     .option("authentication.basic.username", NEO4J_USERNAME)
#     .option("authentication.basic.password", NEO4J_PASSWORD)
#     .option("batch.size", 2500)
#     .option("transaction.retries", "10")
#     .option("relationship.save.strategy", "keys")
#     .option("relationship", "complete")
#     .option("relationship.source.save.mode", "Match")
#     .option("relationship.source.labels", ":entity")
#     .option("relationship.source.node.keys", "ent_id1:ID")
#     .option("relationship.properties", "distance")
#     .option("relationship.target.save.mode", "Match")
#     .option("relationship.target.labels", ":event")
#     .option("relationship.target.node.keys", "eve_id:ID")
#     .mode("Overwrite")
#     .save()
# )

In [22]:
spark.stop()